# 04 · Quality report
One place that proves where every row went: row counts per layer, reconciliation between layers,
data quality results per rule, Delta table history, and time travel.

This notebook only **reads** the lakehouse. Run notebooks 01–03 (or the pipeline) first.

In [1]:
from pathlib import Path
from pyspark.sql import functions as F
from food_pipeline.spark_session import get_spark
from food_pipeline import quality, silver as silver_fns

ROOT = Path.cwd().parent
LAKE = ROOT / "lakehouse"
spark = get_spark()

PATHS = {
    "bronze.survey_raw": LAKE / "bronze" / "survey_raw",
    "quarantine.survey": LAKE / "quarantine" / "survey",
    "silver.survey": LAKE / "silver" / "survey",
    "gold.dim_location": LAKE / "gold" / "dim_location",
    "gold.fact_survey_response": LAKE / "gold" / "fact_survey_response",
    "gold.agg_order_rate_by_segment": LAKE / "gold" / "agg_order_rate_by_segment",
    "gold.agg_area_summary": LAKE / "gold" / "agg_area_summary",
}
PATHS = {name: p.as_posix() for name, p in PATHS.items()}
t = {name: spark.read.format("delta").load(p) for name, p in PATHS.items()}

checks = []  # (check, expected, actual) collected through the notebook, summarised at the end
def check(name, expected, actual):
    checks.append((name, str(expected), str(actual), "PASS" if expected == actual else "FAIL"))
    print(f"{'PASS' if expected == actual else 'FAIL'}  {name}: expected {expected}, got {actual}")

## 1 · Row counts per layer

In [2]:
counts = {name: df.count() for name, df in t.items()}
spark.createDataFrame(list(counts.items()), "table string, rows long").show(truncate=False)

+------------------------------+----+
|table                         |rows|
+------------------------------+----+
|bronze.survey_raw             |388 |
|quarantine.survey             |0   |
|silver.survey                 |285 |
|gold.dim_location             |77  |
|gold.fact_survey_response     |285 |
|gold.agg_order_rate_by_segment|26  |
|gold.agg_area_summary         |77  |
+------------------------------+----+



## 2 · Reconciliation: Bronze → Silver
Rebuild the expected Silver independently from Bronze (clean → DQ → distinct hashes) and compare.
**Bronze = Silver + duplicates removed + quarantined.**

In [3]:
bronze, silver, quarantine = t["bronze.survey_raw"], t["silver.survey"], t["quarantine.survey"]

valid, _ = quality.split_valid(silver_fns.clean(bronze))
expected_hashes = silver_fns.add_record_hash(valid).select("record_hash").distinct()
n_bronze, n_silver, n_quarantine = counts["bronze.survey_raw"], counts["silver.survey"], counts["quarantine.survey"]
n_duplicates = valid.count() - expected_hashes.count()

print(f"bronze {n_bronze} = silver {n_silver} + duplicates {n_duplicates} + quarantined {n_quarantine}\n")
check("bronze = silver + duplicates + quarantined", n_bronze, n_silver + n_duplicates + n_quarantine)
check("silver hashes missing vs bronze", 0, expected_hashes.join(silver, "record_hash", "left_anti").count())
check("silver hashes not explained by bronze", 0, silver.join(expected_hashes, "record_hash", "left_anti").count())
check("record_hash unique in silver", n_silver, silver.select("record_hash").distinct().count())

bronze 388 = silver 285 + duplicates 103 + quarantined 0

PASS  bronze = silver + duplicates + quarantined: expected 388, got 388


PASS  silver hashes missing vs bronze: expected 0, got 0


PASS  silver hashes not explained by bronze: expected 0, got 0


PASS  record_hash unique in silver: expected 285, got 285


### Per batch
Silver keeps the lineage of the *first* batch an answer set arrived in, so batch_02 contributes fewer new rows.

In [4]:
(bronze.groupBy("_batch_id").agg(F.count("*").alias("bronze_rows"))
    .join(silver.groupBy("_batch_id").agg(F.count("*").alias("new_in_silver")), "_batch_id", "left")
    .join(quarantine.groupBy("_batch_id").agg(F.count("*").alias("quarantined")), "_batch_id", "left")
    .fillna(0).orderBy("_batch_id").show())

+---------+-----------+-------------+-----------+
|_batch_id|bronze_rows|new_in_silver|quarantined|
+---------+-----------+-------------+-----------+
| batch_01|        200|          177|          0|
| batch_02|        188|          108|          0|
+---------+-----------+-------------+-----------+



## 3 · Data quality results per rule
Counts come from re-checking Bronze, so rules with zero failures are listed too.

In [5]:
cleaned = silver_fns.clean(bronze)
rule_counts = quality.failure_counts(cleaned)
spark.createDataFrame(list(rule_counts.items()), "rule string, failed_rows long").show(truncate=False)

n_failing = quality.add_failed_rules(cleaned).filter(F.size("failed_rules") > 0).count()
check("quarantine rows = bronze rows failing any rule", n_failing, n_quarantine)

+------------------------+-----------+
|rule                    |failed_rows|
+------------------------+-----------+
|age_18_to_80            |0          |
|family_size_1_to_15     |0          |
|pin_code_bengaluru      |0          |
|latitude_in_bengaluru   |0          |
|longitude_in_bengaluru  |0          |
|income_band_known       |0          |
|ordered_again_known     |0          |
|raw_extra_matches_output|0          |
|gender_allowed          |0          |
|marital_status_allowed  |0          |
|occupation_allowed      |0          |
|customer_type_allowed   |0          |
|feedback_allowed        |0          |
+------------------------+-----------+



PASS  quarantine rows = bronze rows failing any rule: expected 0, got 0


## 4 · Silver → Gold

In [6]:
fact, dim = t["gold.fact_survey_response"], t["gold.dim_location"]
check("fact rows = silver rows", n_silver, counts["gold.fact_survey_response"])
check("silver respondents missing from fact", 0, silver.join(fact, "record_hash", "left_anti").count())
check("fact rows without a location", 0, fact.join(dim, "location_key", "left_anti").count())
check("dim_location rows = distinct silver pin codes", silver.select("pin_code").distinct().count(), counts["gold.dim_location"])
check("area summary respondents add up to fact rows", n_silver, t["gold.agg_area_summary"].agg(F.sum("respondents")).first()[0])
seg = t["gold.agg_order_rate_by_segment"].groupBy("segment_type").agg(F.sum("respondents").alias("total"))
check("every segment type adds up to fact rows", True, all(r["total"] == n_silver for r in seg.collect()))

PASS  fact rows = silver rows: expected 285, got 285


PASS  silver respondents missing from fact: expected 0, got 0


PASS  fact rows without a location: expected 0, got 0
PASS  dim_location rows = distinct silver pin codes: expected 77, got 77


PASS  area summary respondents add up to fact rows: expected 285, got 285


PASS  every segment type adds up to fact rows: expected True, got True


## 5 · Delta table history
Every write is a version in `_delta_log`. Skipped reruns and no-op merges add no versions.

In [7]:
for name in ["bronze.survey_raw", "silver.survey"]:
    print(name)
    (spark.sql(f"DESCRIBE HISTORY delta.`{PATHS[name]}`")
        .select("version", "timestamp", "operation", "operationParameters.mode",
                "operationMetrics.numOutputRows", "operationMetrics.numTargetRowsInserted")
        .orderBy("version").show(truncate=False))

bronze.survey_raw


+-------+-----------------------+---------+------+-------------+---------------------+
|version|timestamp              |operation|mode  |numOutputRows|numTargetRowsInserted|
+-------+-----------------------+---------+------+-------------+---------------------+
|0      |2026-10-26 22:21:26.299|WRITE    |Append|200          |NULL                 |
|1      |2026-10-26 22:35:43.537|WRITE    |Append|188          |NULL                 |
+-------+-----------------------+---------+------+-------------+---------------------+

silver.survey


+-------+-----------------------+---------+-------------+-------------+---------------------+
|version|timestamp              |operation|mode         |numOutputRows|numTargetRowsInserted|
+-------+-----------------------+---------+-------------+-------------+---------------------+
|0      |2026-10-27 15:39:43.189|WRITE    |ErrorIfExists|177          |NULL                 |
|1      |2026-10-27 15:39:55.873|MERGE    |NULL         |108          |108                  |
+-------+-----------------------+---------+-------------+-------------+---------------------+



## 6 · Time travel
Read Silver as it was after each version. Version 0 is Silver after batch_01 only.

In [8]:
silver_versions = spark.sql(f"DESCRIBE HISTORY delta.`{PATHS['silver.survey']}`").select("version").collect()
for v in sorted(r["version"] for r in silver_versions):
    snapshot = spark.read.format("delta").option("versionAsOf", v).load(PATHS["silver.survey"])
    by_batch = {r["_batch_id"]: r["count"] for r in snapshot.groupBy("_batch_id").count().collect()}
    print(f"silver version {v}: {snapshot.count()} rows {dict(sorted(by_batch.items()))}")

v0 = spark.read.format("delta").option("versionAsOf", 0).load(PATHS["silver.survey"])
check("silver version 0 holds only batch_01", ["batch_01"], sorted(r["_batch_id"] for r in v0.select("_batch_id").distinct().collect()))

silver version 0: 177 rows {'batch_01': 177}


silver version 1: 285 rows {'batch_01': 177, 'batch_02': 108}


PASS  silver version 0 holds only batch_01: expected ['batch_01'], got ['batch_01']


## Summary

In [9]:
summary = spark.createDataFrame(checks, "check string, expected string, actual string, status string")
summary.show(truncate=False)
failed = summary.filter("status = 'FAIL'").count()
print(f"{len(checks) - failed} of {len(checks)} checks passed")

+----------------------------------------------+------------+------------+------+
|check                                         |expected    |actual      |status|
+----------------------------------------------+------------+------------+------+
|bronze = silver + duplicates + quarantined    |388         |388         |PASS  |
|silver hashes missing vs bronze               |0           |0           |PASS  |
|silver hashes not explained by bronze         |0           |0           |PASS  |
|record_hash unique in silver                  |285         |285         |PASS  |
|quarantine rows = bronze rows failing any rule|0           |0           |PASS  |
|fact rows = silver rows                       |285         |285         |PASS  |
|silver respondents missing from fact          |0           |0           |PASS  |
|fact rows without a location                  |0           |0           |PASS  |
|dim_location rows = distinct silver pin codes |77          |77          |PASS  |
|area summary re

12 of 12 checks passed
